# CardiLearn v0.5 — Locked Real-Data Cardiac Validation (Colab / T4)

This notebook is the **scientific-validation runner** for Virelion-CardiLearn.

It is intentionally stricter than a demo notebook:

- accepts a locked `h5ad` or `npz + metadata` bundle;
- requires resolved biological hierarchy `study → subject → sample → cell/nucleus`;
- creates a reproducible study-family split and audits hierarchy leakage;
- selects genes using **training observations only**;
- trains the current `CardiLearnResearch` model on a T4 with sparse mini-batch loading;
- freezes the learned representation before downstream evaluation;
- aggregates cells/nuclei to biological sample before inference;
- compares against PCA and a small autoencoder baseline;
- accepts externally generated frozen embeddings for GeneFormer/scGPT/UCE/NicheFormer/Scimilarity/scVI when supplied;
- evaluates held-out injury and maturation, including bootstrap CIs and a grouped permutation null;
- reports cross-study transfer and only calls a test species “species-unseen” when that species was absent from model training;
- writes hashes, configuration, split information, runtime information, results, and model artifacts.

**No dataset or biological result is fabricated by this notebook.**


In [ ]:
import os, sys, subprocess, json, hashlib, platform, time, textwrap, shutil, warnings
from pathlib import Path

REPO_URL = "https://github.com/Virelion-Biotech/Virelion-CardiLearn.git"
REPO_DIR = Path("/content/Virelion-CardiLearn")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[torch,bio,bench]"],
    cwd=REPO_DIR,
    check=True,
)
print("Repo:", REPO_DIR)
print("Python:", sys.version.split()[0])


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. Configuration

The notebook expects one of these input layouts:

**Option A — AnnData**
`DATA_DIR/expression.h5ad`

The file should contain raw counts in `adata.raw.X`, `adata.layers[RAW_LAYER]`, or `adata.X`. The required metadata columns live in `adata.obs`.

**Option B — matrix + metadata**
`DATA_DIR/expression.npz` with arrays `X` and `genes`, plus `metadata.parquet` or `metadata.csv`.

Required metadata:

`study_id, subject_id, sample_id, species, assay, cell_type, maturation, injury`

Optional but strongly recommended:

`study_family_id, tissue`

For related accessions that are not independent experiments, populate `study_family_id` yourself. The notebook will not infer biological independence.


In [ ]:
DATA_DIR = Path("/content/cardiLearn_data")

# Optional: point this at a ZIP on Google Drive. Leave empty when DATA_DIR already contains the bundle.
DATA_ZIP = ""  # e.g. "/content/drive/MyDrive/cardiLearn_locked_bundle.zip"

# AnnData expression source. Set only when a specific raw-count layer should be used.
RAW_LAYER = None        # e.g. "counts"
USE_ADATA_RAW = True

# First T4 run defaults. Increase only after the first run is stable.
N_GENES = 5000
BATCH_SIZE = 4
GRAD_ACCUMULATION = 8
EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
MASK_FRACTION = 0.15
SEED = 42

# Baseline dimensions / training.
BASELINE_DIM = 64
AE_HIDDEN = 256
AE_LATENT = 64
AE_EPOCHS = 5
AE_BATCH_SIZE = 128

# Statistics.
N_PERMUTATIONS = 1000
N_BOOTSTRAP = 2000

# External frozen embedding folder.
EXTERNAL_EMBEDDINGS_DIR = DATA_DIR / "external_embeddings"

# Save large cell-level embeddings only when explicitly needed.
SAVE_CELL_EMBEDDINGS = False

# Optional GitHub result push. This pushes only lightweight result/provenance files by default.
PUSH_RESULTS = False
GITHUB_REPO = "Virelion-Biotech/Virelion-CardiLearn"

RUN_ID = time.strftime("realdata-%Y%m%dT%H%M%SZ", time.gmtime())
RESULTS_DIR = Path("/content") / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ID:", RUN_ID)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
if DATA_ZIP:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(DATA_ZIP, DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"{DATA_DIR} does not exist. Put the locked data bundle there or set DATA_ZIP."
    )

print("Input files:")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(DATA_DIR), f"({p.stat().st_size/1024**2:.1f} MB)")


## 2. Load the locked expression matrix and metadata

The notebook deliberately preserves sparse matrices whenever possible. Only individual mini-batches are densified for the PyTorch model.

A non-integer expression matrix is rejected by default because the research decoder is a negative-binomial count model. If the source uses transformed values, stop here and provide the actual raw-count layer instead.


In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def load_bundle(data_dir):
    h5ad = data_dir / "expression.h5ad"
    npz = data_dir / "expression.npz"
    if h5ad.exists():
        import anndata as ad
        adata = ad.read_h5ad(h5ad, backed=None)
        if USE_ADATA_RAW and getattr(adata, "raw", None) is not None:
            X = adata.raw.X
            genes = [str(x) for x in adata.raw.var_names]
            source = "adata.raw.X"
        elif RAW_LAYER and RAW_LAYER in adata.layers:
            X = adata.layers[RAW_LAYER]
            genes = [str(x) for x in adata.var_names]
            source = f"adata.layers[{RAW_LAYER!r}]"
        else:
            X = adata.X
            genes = [str(x) for x in adata.var_names]
            source = "adata.X"
        metadata = adata.obs.copy().reset_index(names="cell_id")
        return X, genes, metadata, h5ad, source
    if npz.exists():
        with np.load(npz, allow_pickle=False) as payload:
            if "X" not in payload or "genes" not in payload:
                raise ValueError("expression.npz must contain X and genes arrays")
            X = payload["X"]
            genes = [str(x) for x in payload["genes"].tolist()]
        meta_path = data_dir / "metadata.parquet"
        if not meta_path.exists():
            meta_path = data_dir / "metadata.csv"
        if not meta_path.exists():
            raise FileNotFoundError("Need metadata.parquet or metadata.csv beside expression.npz")
        metadata = pd.read_parquet(meta_path) if meta_path.suffix == ".parquet" else pd.read_csv(meta_path)
        metadata = metadata.reset_index(drop=True)
        if "cell_id" not in metadata.columns:
            metadata.insert(0, "cell_id", [f"cell_{i}" for i in range(len(metadata))])
        return X, genes, metadata, npz, "npz:X"
    raise FileNotFoundError("Expected expression.h5ad or expression.npz in DATA_DIR")

X, genes, metadata, expression_source_path, expression_source = load_bundle(DATA_DIR)

if len(metadata) != X.shape[0]:
    raise ValueError(f"Rows mismatch: expression has {X.shape[0]} observations but metadata has {len(metadata)}")
if len(set(genes)) != len(genes):
    raise ValueError("Gene names must be unique")
if X.ndim != 2 or X.shape[0] < 3 or X.shape[1] < 10:
    raise ValueError("Expression matrix is too small or not 2-D")

print("Expression source:", expression_source)
print("Shape:", X.shape, "sparse=" + str(sparse.issparse(X)))
print("Metadata columns:", list(metadata.columns))


In [ ]:
REQUIRED = {
    "study_id", "subject_id", "sample_id", "species",
    "assay", "cell_type", "maturation", "injury",
}
missing = sorted(REQUIRED - set(metadata.columns))
if missing:
    raise ValueError(f"Missing required metadata columns: {missing}")

for col in ["study_id", "subject_id", "sample_id", "species", "assay", "cell_type", "injury"]:
    s = metadata[col].astype(str).str.strip()
    if s.eq("").any() or s.eq("nan").any() or s.eq("None").any():
        raise ValueError(f"{col} contains unresolved/empty values")

metadata["maturation"] = pd.to_numeric(metadata["maturation"], errors="coerce")
if metadata["maturation"].isna().any():
    bad = metadata.loc[metadata["maturation"].isna(), "cell_id"].astype(str).head(10).tolist()
    raise ValueError(
        "maturation must be numeric (or manually mapped before this cell). "
        f"Examples with unresolved values: {bad}"
    )

# Hierarchy identity checks.
for child, parent in [("sample_id", "subject_id"), ("subject_id", "study_id")]:
    counts = metadata.groupby(child)[parent].nunique()
    if (counts > 1).any():
        bad = counts[counts > 1].index.tolist()[:10]
        raise ValueError(f"{child} maps to multiple {parent} values: {bad}")

# Related-accession grouping: do not infer it.
if "study_family_id" not in metadata.columns:
    metadata["study_family_id"] = metadata["study_id"].astype(str)

# Make IDs explicit strings.
for col in ["cell_id", "study_id", "study_family_id", "subject_id", "sample_id", "species", "assay", "cell_type", "injury"]:
    metadata[col] = metadata[col].astype(str)

# Raw-count sanity check on a bounded sample.
rng = np.random.default_rng(SEED)
row_idx = rng.choice(X.shape[0], size=min(256, X.shape[0]), replace=False)
if sparse.issparse(X):
    sample = X[row_idx].toarray()
else:
    sample = np.asarray(X)[row_idx]
finite = np.isfinite(sample).all()
nonnegative = (sample >= 0).all()
integer_fraction = float(np.mean(np.isclose(sample, np.rint(sample), atol=1e-5)))
print("finite:", finite, "nonnegative:", nonnegative, "integer-like fraction:", integer_fraction)
if not finite or not nonnegative:
    raise ValueError("Raw expression contains non-finite or negative values")
if integer_fraction < 0.995:
    raise ValueError(
        "Expression does not look like raw counts (integer-like fraction < 0.995). "
        "Select the true raw-count layer instead of training on transformed values."
    )

print("\nSpecies:")
print(metadata["species"].value_counts())
print("\nStudies:", metadata["study_id"].nunique(),
      "Study families:", metadata["study_family_id"].nunique(),
      "Subjects:", metadata["subject_id"].nunique(),
      "Samples:", metadata["sample_id"].nunique(),
      "Observations:", len(metadata))


## 3. Lock the split and data provenance

The split unit is `study_family_id`. The notebook uses the repository's existing split implementation, then verifies that study, subject, sample, and study-family IDs occur in exactly one partition.

The resulting `data_lock.json` is the scientific input record. Do not modify the data or split after seeing test performance.


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from cardilearn.prototype.splits import study_split, assign_split, assert_no_hierarchy_leakage
from cardilearn.reproducibility import dataframe_fingerprint, fingerprint_ids

splits = study_split(
    metadata,
    seed=SEED,
    train_fraction=0.625,
    val_fraction=0.125,
    group_column="study_family_id",
)
metadata = assign_split(metadata, splits, group_column="study_family_id")
assert_no_hierarchy_leakage(metadata)

if metadata["_split"].isna().any():
    raise AssertionError("Unassigned observations remain")

split_summary = (
    metadata.groupby(["_split", "species"])
    .size()
    .rename("n_observations")
    .reset_index()
)
sample_split_summary = (
    metadata[["sample_id", "_split"]].drop_duplicates()
    .groupby("_split").size().rename("n_samples").reset_index()
)

lock = {
    "schema_version": "cardilearn-realdata-lock-v1",
    "run_id": RUN_ID,
    "repo": GITHUB_REPO,
    "repo_commit": subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "expression_source": str(expression_source_path),
    "expression_source_sha256": sha256_file(expression_source_path),
    "expression_source_selector": expression_source,
    "expression_shape": list(map(int, X.shape)),
    "gene_namespace": "source_var_names",
    "metadata_fingerprint": dataframe_fingerprint(metadata),
    "split_fingerprint": fingerprint_ids([
        f"{k}:{v}" for k, values in splits.items() if k != "group_column" for v in values
    ]),
    "splits": splits,
    "seed": SEED,
    "hierarchy": "study_family_id > study_id > subject_id > sample_id > cell_id",
    "scientific_note": "Lock this file before inspecting held-out test performance.",
}
(RESULTS_DIR / "data_lock.json").write_text(json.dumps(lock, indent=2, sort_keys=True) + "\n")

print(json.dumps(splits, indent=2))
display(split_summary)
display(sample_split_summary)


## 4. Training-only gene selection

We select the highest-variance genes using **only cells in the training partition**. The selected indices are then frozen and applied to validation/test observations.

This avoids test-derived feature selection.


In [ ]:
def select_genes_train_only_sparse(X, train_mask, n_genes):
    if n_genes < 1 or n_genes > X.shape[1]:
        raise ValueError("N_GENES must be between 1 and the full gene count")
    train_X = X[train_mask]
    n = train_X.shape[0]
    if sparse.issparse(train_X):
        sums = np.asarray(train_X.sum(axis=0)).ravel().astype(np.float64)
        sq_sums = np.asarray(train_X.multiply(train_X).sum(axis=0)).ravel().astype(np.float64)
    else:
        arr = np.asarray(train_X, dtype=np.float64)
        sums = arr.sum(axis=0)
        sq_sums = np.square(arr).sum(axis=0)
    var = np.maximum(sq_sums / n - np.square(sums / n), 0.0)
    selected = np.argsort(var, kind="stable")[-n_genes:]
    return np.sort(selected), var

train_mask = metadata["_split"].eq("train").to_numpy()
selected_idx, train_gene_variance = select_genes_train_only_sparse(X, train_mask, min(N_GENES, X.shape[1]))
selected_genes = [genes[i] for i in selected_idx]

if len(selected_genes) != len(set(selected_genes)):
    raise AssertionError("Selected genes are not unique")

np.savez_compressed(RESULTS_DIR / "selected_genes.npz", indices=selected_idx, genes=np.asarray(selected_genes, dtype=str))
(RESULTS_DIR / "selected_genes.txt").write_text("\n".join(selected_genes) + "\n")

print("Selected genes:", len(selected_genes))
print("Training observations:", int(train_mask.sum()))
print("Validation observations:", int(metadata["_split"].eq("validation").sum()))
print("Test observations:", int(metadata["_split"].eq("test").sum()))


## 5. Encode biological metadata using training-only vocabularies

Categorical vocabularies are fit on training observations only. Unseen validation/test categories map to the explicit `<UNK>` code.

The injury target is binary. Recognized labels map `sham/control/... → 0` and `MI/injury/infarct/... → 1`; otherwise the two training labels are deterministically ordered.


In [ ]:
from cardilearn.torch_training import CategoryEncoder

def build_injury_mapping(values):
    labels = sorted(pd.Series(values).astype(str).unique().tolist())
    if len(labels) != 2:
        raise ValueError(f"Injury training labels must be binary; found {labels}")
    norm = {x: x.strip().lower() for x in labels}
    pos_terms = {"mi", "injured", "injury", "infarct", "infarction", "ischemia", "ischaemia"}
    neg_terms = {"sham", "control", "uninjured", "healthy", "naive", "vehicle"}
    pos = [x for x in labels if norm[x] in pos_terms]
    neg = [x for x in labels if norm[x] in neg_terms]
    if len(pos) == 1 and len(neg) == 1:
        return {neg[0]: 0, pos[0]: 1}
    return {x: i for i, x in enumerate(labels)}

train_meta = metadata.loc[train_mask].copy()
species_enc = CategoryEncoder(train_meta["species"])
assay_enc = CategoryEncoder(train_meta["assay"])
cell_enc = CategoryEncoder(train_meta["cell_type"])
injury_map = build_injury_mapping(train_meta["injury"])

encoded = metadata.copy()
encoded["species_code"] = species_enc.encode(encoded["species"])
encoded["assay_code"] = assay_enc.encode(encoded["assay"])
cell_code_raw = cell_enc.encode(encoded["cell_type"])
encoded["cell_type_code"] = np.where(cell_code_raw == 0, -1, cell_code_raw)
encoded["injury_code"] = encoded["injury"].map(injury_map).fillna(-1).astype(np.int64)

# Stable regression target scaling fitted on train only.
maturation_mean = float(train_meta["maturation"].mean())
maturation_std = float(train_meta["maturation"].std(ddof=0))
if not np.isfinite(maturation_std) or maturation_std == 0:
    maturation_std = 1.0
encoded["maturation_z"] = (
    (encoded["maturation"].astype(float) - maturation_mean) / maturation_std
).astype(np.float32)

# Check target availability.
if encoded.loc[train_mask, "injury_code"].nunique() != 2:
    raise ValueError("Training partition must contain both injury classes")

metadata = encoded

vocab = {
    "species": species_enc.to_dict(),
    "assay": assay_enc.to_dict(),
    "cell_type": cell_enc.to_dict(),
    "injury": injury_map,
    "maturation_mean_train": maturation_mean,
    "maturation_std_train": maturation_std,
}
(RESULTS_DIR / "metadata_vocab.json").write_text(json.dumps(vocab, indent=2, sort_keys=True) + "\n")
display(pd.DataFrame({
    "injury_label": list(injury_map.keys()),
    "encoded": list(injury_map.values()),
}))


## 6. Build the sparse mini-batch loader

Only individual sparse rows are densified for each PyTorch batch. This is intentionally different from the legacy dense CLI runner and is the path to use for real single-cell/single-nucleus matrices.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SparseResearchDataset(Dataset):
    def __init__(self, X, row_ids, meta, selected_idx):
        self.X = X
        self.row_ids = np.asarray(row_ids, dtype=np.int64)
        self.meta = meta.iloc[self.row_ids].reset_index(drop=True)
        self.selected_idx = np.asarray(selected_idx, dtype=np.int64)

    def __len__(self):
        return len(self.row_ids)

    def _dense_row(self, local_index):
        row = self.X[self.row_ids[local_index], self.selected_idx]
        if sparse.issparse(row):
            return np.asarray(row.toarray()).ravel().astype(np.float32)
        return np.asarray(row, dtype=np.float32).ravel()

    def __getitem__(self, i):
        return {
            "counts": torch.from_numpy(self._dense_row(i)),
            "species": torch.tensor(int(self.meta.iloc[i]["species_code"]), dtype=torch.long),
            "assay": torch.tensor(int(self.meta.iloc[i]["assay_code"]), dtype=torch.long),
            "cell_type": torch.tensor(int(self.meta.iloc[i]["cell_type_code"]), dtype=torch.long),
            "maturation": torch.tensor(float(self.meta.iloc[i]["maturation_z"]), dtype=torch.float32),
            "injury": torch.tensor(float(self.meta.iloc[i]["injury_code"]), dtype=torch.float32),
        }

train_rows = np.flatnonzero(metadata["_split"].eq("train").to_numpy())
train_dataset = SparseResearchDataset(X, train_rows, metadata, selected_idx)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Training batches:", len(train_loader))
print("Batch example keys:", train_dataset[0].keys())


## 7. Train CardiLearnResearch on the T4

The model is trained **only on the training split**.

The checkpoint is saved every epoch and the final model state is copied to `cardilearn_model.pt`.

The objective schedule is the repository's research schedule: representation reconstruction/masking first, followed by biological-state heads and contrastive/VICReg terms.


In [ ]:
from cardilearn.research_model import CardiLearnResearch
from cardilearn.objectives import ObjectiveSchedule, ObjectiveStage, ObjectiveWeights
from cardilearn.torch_training import TorchTrainConfig, fit_research_model, history_to_json

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_species = max(species_enc.to_dict().values()) + 1
n_assays = max(assay_enc.to_dict().values()) + 1
n_cell_types = max(cell_enc.to_dict().values()) + 1

model = CardiLearnResearch(
    n_genes=len(selected_idx),
    n_species=n_species,
    n_assays=n_assays,
    n_cell_types=n_cell_types,
    gene_dim=256,
    n_programs=128,
    n_layers=6,
    n_heads=8,
    shared_dim=384,
    private_dim=128,
    decoder_dim=256,
    router_chunk_size=2048,
).to(device)

train_cfg = TorchTrainConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    grad_accumulation_steps=GRAD_ACCUMULATION,
    gradient_clip_norm=5.0,
    mask_fraction=MASK_FRACTION,
    mixed_precision=True,
    seed=SEED,
    device="auto",
)

schedule = ObjectiveSchedule((
    ObjectiveStage(
        "representation",
        ObjectiveWeights(
            reconstruction=1.0, masked=1.0, contrastive=0.0, vicreg=0.10,
            cell_type=0.0, maturation=0.0, injury=0.0, species_adversarial=0.0
        ),
    ),
    ObjectiveStage(
        "biological_state",
        ObjectiveWeights(
            reconstruction=1.0, masked=1.0, contrastive=0.25, vicreg=0.10,
            cell_type=0.50, maturation=1.0, injury=0.50, species_adversarial=0.0
        ),
    ),
))

train_start = time.time()
history = fit_research_model(
    model,
    train_loader,
    config=train_cfg,
    schedule=schedule,
    checkpoint_dir=RESULTS_DIR / "checkpoints",
)
train_seconds = time.time() - train_start

torch.save(
    {
        "schema_version": "cardilearn-realdata-v1",
        "model_state_dict": model.state_dict(),
        "model_config": {
            "n_genes": int(len(selected_idx)),
            "n_species": int(n_species),
            "n_assays": int(n_assays),
            "n_cell_types": int(n_cell_types),
            "gene_dim": 256,
            "n_programs": 128,
            "n_layers": 6,
            "n_heads": 8,
            "shared_dim": 384,
            "private_dim": 128,
            "decoder_dim": 256,
            "router_chunk_size": 2048,
        },
        "selected_genes": selected_genes,
        "vocab": vocab,
        "seed": SEED,
    },
    RESULTS_DIR / "cardilearn_model.pt",
)
history_to_json(history, RESULTS_DIR / "training_history.json")

training_summary = {
    "device": str(device),
    "cuda_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "parameter_count": int(model.parameter_count()),
    "seconds": float(train_seconds),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUMULATION,
}
(RESULTS_DIR / "training_summary.json").write_text(json.dumps(training_summary, indent=2, sort_keys=True) + "\n")
print(json.dumps(training_summary, indent=2))


## 8. Freeze the representation and encode all observations

No gradient updates happen below this point.

Cell/nucleus embeddings are generated in sparse mini-batches, then aggregated to biological sample. The primary benchmark therefore treats each biological sample as one inferential unit.


In [ ]:
def iter_sparse_batches(X, row_ids, selected_idx, batch_size=128):
    row_ids = np.asarray(row_ids, dtype=np.int64)
    for start in range(0, len(row_ids), batch_size):
        ids = row_ids[start:start+batch_size]
        batch = X[ids][:, selected_idx]
        if sparse.issparse(batch):
            batch = batch.toarray()
        yield ids, np.asarray(batch, dtype=np.float32)

def encode_sparse_model(model, X, metadata, selected_idx, batch_size=32):
    model.eval()
    all_z = []
    all_ids = []
    with torch.no_grad():
        for ids, batch in iter_sparse_batches(X, np.arange(X.shape[0]), selected_idx, batch_size=batch_size):
            species = torch.tensor(metadata.iloc[ids]["species_code"].to_numpy(), dtype=torch.long, device=device)
            assay = torch.tensor(metadata.iloc[ids]["assay_code"].to_numpy(), dtype=torch.long, device=device)
            counts = torch.tensor(batch, dtype=torch.float32, device=device)
            _, z_shared, _, _ = model.encode(counts, species, assay)
            all_z.append(z_shared.detach().cpu().numpy().astype(np.float32))
            all_ids.append(ids)
    return np.concatenate(all_z, axis=0)

encode_start = time.time()
cell_z = encode_sparse_model(model, X, metadata, selected_idx, batch_size=max(BATCH_SIZE, 16))
encode_seconds = time.time() - encode_start

if not np.isfinite(cell_z).all():
    raise ValueError("Non-finite CardiLearn embeddings produced")

metadata["cell_embedding_row"] = np.arange(len(metadata))
print("Cell embedding shape:", cell_z.shape, "seconds:", round(encode_seconds, 2))

if SAVE_CELL_EMBEDDINGS:
    np.savez_compressed(
        RESULTS_DIR / "cardilearn_cell_embeddings.npz",
        embeddings=cell_z,
        cell_id=metadata["cell_id"].to_numpy(dtype=str),
        sample_id=metadata["sample_id"].to_numpy(dtype=str),
    )


In [ ]:
def aggregate_cells_to_samples(z, meta, group_col="sample_id"):
    rows = []
    records = []
    for sample_id, idx in meta.groupby(group_col, sort=False).indices.items():
        idx = np.asarray(idx, dtype=int)
        sample_meta = meta.iloc[idx]
        for target in ["injury_code", "maturation", "species", "study_id", "study_family_id", "subject_id"]:
            vals = sample_meta[target].astype(str if target not in ["injury_code", "maturation"] else float).to_numpy()
            if target == "maturation":
                if not np.allclose(vals.astype(float), vals.astype(float)[0], equal_nan=True):
                    raise ValueError(f"Sample {sample_id} has non-constant maturation")
            else:
                if len(np.unique(vals)) != 1:
                    raise ValueError(f"Sample {sample_id} has multiple {target} values")
        rows.append(z[idx].mean(axis=0))
        records.append({
            "sample_id": str(sample_id),
            "study_id": str(sample_meta["study_id"].iloc[0]),
            "study_family_id": str(sample_meta["study_family_id"].iloc[0]),
            "subject_id": str(sample_meta["subject_id"].iloc[0]),
            "species": str(sample_meta["species"].iloc[0]),
            "split": str(sample_meta["_split"].iloc[0]),
            "injury": float(sample_meta["injury_code"].iloc[0]),
            "maturation": float(sample_meta["maturation"].iloc[0]),
            "n_cells": int(len(idx)),
        })
    return np.asarray(rows, dtype=np.float32), pd.DataFrame(records)

sample_z, sample_meta = aggregate_cells_to_samples(cell_z, metadata)

if sample_meta["sample_id"].duplicated().any():
    raise AssertionError("Duplicate sample IDs after aggregation")

sample_z_fp = hashlib.sha256(np.asarray(sample_z, dtype=np.float32).tobytes()).hexdigest()
np.savez_compressed(
    RESULTS_DIR / "cardilearn_sample_embeddings.npz",
    embeddings=sample_z,
    sample_id=sample_meta["sample_id"].to_numpy(dtype=str),
)
sample_meta.to_parquet(RESULTS_DIR / "sample_metadata.parquet", index=False)

print("Sample embedding shape:", sample_z.shape)
display(sample_meta.groupby("split").size().rename("n_samples"))
display(sample_meta.groupby(["split", "species"]).size().rename("n_samples"))


## 9. PCA baseline, fitted only on training cells

PCA is fitted using incremental batches to avoid requiring the complete sparse matrix in RAM. The input is library-size normalized `log1p` counts.

This produces frozen cell embeddings which are aggregated to sample with the same procedure used for CardiLearn.


In [ ]:
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler

def normalized_log_batch(counts, target_sum=1e4):
    counts = np.asarray(counts, dtype=np.float32)
    lib = np.maximum(counts.sum(axis=1, keepdims=True), 1.0)
    return np.log1p((counts / lib) * target_sum).astype(np.float32)

pca_dim = min(BASELINE_DIM, len(selected_idx))
pca = IncrementalPCA(n_components=pca_dim, batch_size=256)

for ids, batch in iter_sparse_batches(
    X,
    train_rows,
    selected_idx,
    batch_size=max(256, AE_BATCH_SIZE),
):
    pca.partial_fit(normalized_log_batch(batch))

def transform_pca_all(X, selected_idx, pca, batch_size=256):
    chunks = []
    for _, batch in iter_sparse_batches(X, np.arange(X.shape[0]), selected_idx, batch_size=batch_size):
        chunks.append(pca.transform(normalized_log_batch(batch)).astype(np.float32))
    return np.concatenate(chunks, axis=0)

pca_cell_z = transform_pca_all(X, selected_idx, pca)
pca_sample_z, _ = aggregate_cells_to_samples(pca_cell_z, metadata)
np.savez_compressed(
    RESULTS_DIR / "pca_sample_embeddings.npz",
    embeddings=pca_sample_z,
    sample_id=sample_meta["sample_id"].to_numpy(dtype=str),
)
print("PCA sample embedding:", pca_sample_z.shape)


## 10. Small autoencoder baseline

This is a deliberately modest nonlinear baseline, not a second large foundation model. It is trained on the same training cells, with the same frozen gene selection and normalized-log expression input.


In [ ]:
import torch.nn as nn

class SmallAutoencoder(nn.Module):
    def __init__(self, n_in, hidden=256, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_in),
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

ae = SmallAutoencoder(len(selected_idx), hidden=AE_HIDDEN, latent=AE_LATENT).to(device)
ae_opt = torch.optim.AdamW(ae.parameters(), lr=1e-3, weight_decay=1e-4)

ae_train_start = time.time()
ae_loss_history = []
ae_train_rows = train_rows.copy()

for epoch in range(1, AE_EPOCHS + 1):
    rng = np.random.default_rng(SEED + epoch)
    order = rng.permutation(ae_train_rows)
    epoch_loss = 0.0
    n_seen = 0
    ae.train()
    for start in range(0, len(order), AE_BATCH_SIZE):
        ids = order[start:start+AE_BATCH_SIZE]
        batch = X[ids][:, selected_idx]
        if sparse.issparse(batch):
            batch = batch.toarray()
        x = torch.tensor(normalized_log_batch(batch), dtype=torch.float32, device=device)
        z, recon = ae(x)
        loss = nn.functional.mse_loss(recon, x)
        ae_opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ae.parameters(), 5.0)
        ae_opt.step()
        epoch_loss += float(loss.detach()) * len(ids)
        n_seen += len(ids)
    ae_loss_history.append({"epoch": epoch, "loss": epoch_loss / max(n_seen, 1)})
    print(f"AE epoch {epoch}/{AE_EPOCHS}: {ae_loss_history[-1]['loss']:.6f}")

ae_seconds = time.time() - ae_train_start

def encode_ae_all(X, selected_idx, ae, batch_size=256):
    ae.eval()
    chunks = []
    with torch.no_grad():
        for _, batch in iter_sparse_batches(X, np.arange(X.shape[0]), selected_idx, batch_size=batch_size):
            x = torch.tensor(normalized_log_batch(batch), dtype=torch.float32, device=device)
            z, _ = ae(x)
            chunks.append(z.cpu().numpy().astype(np.float32))
    return np.concatenate(chunks, axis=0)

ae_cell_z = encode_ae_all(X, selected_idx, ae)
ae_sample_z, _ = aggregate_cells_to_samples(ae_cell_z, metadata)
np.savez_compressed(
    RESULTS_DIR / "autoencoder_sample_embeddings.npz",
    embeddings=ae_sample_z,
    sample_id=sample_meta["sample_id"].to_numpy(dtype=str),
)
(RESULTS_DIR / "autoencoder_history.json").write_text(json.dumps(ae_loss_history, indent=2) + "\n")
print("AE sample embedding:", ae_sample_z.shape, "seconds:", round(ae_seconds, 2))


## 11. Fixed-split benchmark evaluator

There is **one locked test set**. Hyperparameters are selected on the validation set, then the probe is refit on train + validation and evaluated once on the locked test set.

Metrics:

- injury classification: AUROC, AUPRC, balanced accuracy, macro-F1;
- maturation regression: MAE, RMSE, R²;
- test uncertainty: 95% bootstrap CI;
- injury permutation null: one-sided empirical p-value.

The bootstrap is performed over biological samples, so a sample with many cells does not receive extra inferential weight.


In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)

def _fixed_split_indices(meta):
    tr = np.flatnonzero(meta["split"].eq("train").to_numpy())
    va = np.flatnonzero(meta["split"].eq("validation").to_numpy())
    te = np.flatnonzero(meta["split"].eq("test").to_numpy())
    if min(len(tr), len(va), len(te)) < 2:
        raise ValueError(f"Too few samples in one split: train={len(tr)}, val={len(va)}, test={len(te)}")
    return tr, va, te

def fit_best_classifier(Z, y, meta):
    tr, va, te = _fixed_split_indices(meta)
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[va])) < 2 or len(np.unique(y[te])) < 2:
        raise ValueError("Injury benchmark requires both classes in train, validation, and test")
    candidates = [0.01, 0.1, 1.0, 10.0, 100.0]
    scores = []
    for C in candidates:
        est = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=C, max_iter=4000, random_state=SEED),
        )
        est.fit(Z[tr], y[tr])
        val_score = est.predict_proba(Z[va])[:, 1]
        scores.append((float(roc_auc_score(y[va], val_score)), C))
    best_auc, best_C = max(scores, key=lambda x: (x[0], -x[1]))
    est = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=best_C, max_iter=4000, random_state=SEED),
    )
    fit_idx = np.concatenate([tr, va])
    est.fit(Z[fit_idx], y[fit_idx])
    score = est.predict_proba(Z[te])[:, 1]
    pred = (score >= 0.5).astype(int)
    metrics = {
        "auroc": float(roc_auc_score(y[te], score)),
        "auprc": float(average_precision_score(y[te], score)),
        "balanced_accuracy": float(balanced_accuracy_score(y[te], pred)),
        "f1_macro": float(f1_score(y[te], pred, average="macro", zero_division=0)),
        "validation_auroc": float(best_auc),
        "selected_C": float(best_C),
    }
    return est, score, pred, metrics, te

def fit_best_regressor(Z, y, meta):
    tr, va, te = _fixed_split_indices(meta)
    candidates = [0.01, 0.1, 1.0, 10.0, 100.0]
    choices = []
    for alpha in candidates:
        est = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        est.fit(Z[tr], y[tr])
        val_pred = est.predict(Z[va])
        mse = mean_squared_error(y[va], val_pred)
        choices.append((float(mse), alpha))
    best_mse, best_alpha = min(choices, key=lambda x: (x[0], x[1]))
    est = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
    fit_idx = np.concatenate([tr, va])
    est.fit(Z[fit_idx], y[fit_idx])
    pred = est.predict(Z[te])
    metrics = {
        "mae": float(mean_absolute_error(y[te], pred)),
        "rmse": float(mean_squared_error(y[te], pred, squared=False)),
        "r2": float(r2_score(y[te], pred)),
        "validation_mse": float(best_mse),
        "selected_alpha": float(best_alpha),
    }
    return est, pred, metrics, te

def bootstrap_auroc(y, score, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y), size=len(y))
        yy = y[idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, score[idx]))
    if not vals:
        return (np.nan, np.nan)
    return (float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975)))

def permutation_pvalue(y, score, n=1000, seed=SEED):
    observed = roc_auc_score(y, score)
    rng = np.random.default_rng(seed)
    ge = 0
    valid = 0
    for _ in range(n):
        yp = rng.permutation(y)
        if len(np.unique(yp)) < 2:
            continue
        null = roc_auc_score(yp, score)
        ge += int(null >= observed)
        valid += 1
    return float(observed), float((1 + ge) / (1 + valid))

def bootstrap_regression(y, pred, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    mae_vals, rmse_vals, r2_vals = [], [], []
    for _ in range(n):
        idx = rng.integers(0, len(y), size=len(y))
        yy, pp = y[idx], pred[idx]
        mae_vals.append(mean_absolute_error(yy, pp))
        rmse_vals.append(mean_squared_error(yy, pp, squared=False))
        if np.var(yy) > 0:
            r2_vals.append(r2_score(yy, pp))
    return {
        "mae_ci_low": float(np.quantile(mae_vals, 0.025)),
        "mae_ci_high": float(np.quantile(mae_vals, 0.975)),
        "rmse_ci_low": float(np.quantile(rmse_vals, 0.025)),
        "rmse_ci_high": float(np.quantile(rmse_vals, 0.975)),
        "r2_ci_low": float(np.quantile(r2_vals, 0.025)) if r2_vals else np.nan,
        "r2_ci_high": float(np.quantile(r2_vals, 0.975)) if r2_vals else np.nan,
    }


In [ ]:
def evaluate_model(name, Z, sample_meta):
    Z = np.asarray(Z, dtype=np.float32)
    if not np.isfinite(Z).all():
        raise ValueError(f"{name}: non-finite embeddings")
    y_injury = sample_meta["injury"].to_numpy(dtype=int)
    y_maturation = sample_meta["maturation"].to_numpy(dtype=float)

    clf, score, pred, injury_metrics, test_idx = fit_best_classifier(Z, y_injury, sample_meta)
    lo, hi = bootstrap_auroc(y_injury[test_idx], score, n=N_BOOTSTRAP, seed=SEED)
    observed, p_null = permutation_pvalue(y_injury[test_idx], score, n=N_PERMUTATIONS, seed=SEED)
    injury_metrics.update({
        "test_auroc_ci_low": lo,
        "test_auroc_ci_high": hi,
        "permutation_pvalue": p_null,
    })

    reg, reg_pred, maturation_metrics, reg_test_idx = fit_best_regressor(Z, y_maturation, sample_meta)
    maturation_metrics.update(bootstrap_regression(
        y_maturation[reg_test_idx], reg_pred, n=N_BOOTSTRAP, seed=SEED
    ))

    return {
        "model": name,
        "injury": injury_metrics,
        "maturation": maturation_metrics,
        "test_samples": int(len(test_idx)),
        "test_species": sorted(sample_meta.iloc[test_idx]["species"].unique().tolist()),
        "test_studies": sorted(sample_meta.iloc[test_idx]["study_id"].unique().tolist()),
    }


## 12. Run the frozen representation matrix

The notebook always evaluates CardiLearn, PCA, and the autoencoder.

External models are only evaluated when the user supplies a valid `.npz` under `external_embeddings/`. The expected file format is:

```
embeddings: [n_rows, dim]
sample_id:  [n_rows]
```

The rows may be cell-level or sample-level. Cell-level rows are averaged to sample using `sample_id`.

Missing external models are reported as **unavailable**, never replaced with synthetic numbers.


In [ ]:
benchmark_payloads = {
    "cardilearn_research": sample_z,
    "pca": pca_sample_z,
    "autoencoder": ae_sample_z,
}

def load_external_embedding_npz(path):
    with np.load(path, allow_pickle=False) as payload:
        if "embeddings" not in payload or "sample_id" not in payload:
            raise ValueError(f"{path.name}: expected embeddings and sample_id arrays")
        Z = np.asarray(payload["embeddings"], dtype=np.float32)
        ids = np.asarray(payload["sample_id"]).astype(str)
    if Z.ndim != 2 or len(Z) != len(ids):
        raise ValueError(f"{path.name}: embeddings/sample_id mismatch")
    if not np.isfinite(Z).all():
        raise ValueError(f"{path.name}: non-finite embeddings")
    frame = pd.DataFrame({"sample_id": ids})
    if frame["sample_id"].duplicated().any():
        # Treat repeated IDs as cell/nucleus-level representations and aggregate.
        rows = []
        unique_ids = pd.unique(ids)
        for sid in unique_ids:
            rows.append(Z[ids == sid].mean(axis=0))
        Z = np.asarray(rows, dtype=np.float32)
        ids = np.asarray(unique_ids, dtype=str)
    return Z, ids

external_status = {}
if EXTERNAL_EMBEDDINGS_DIR.exists():
    for path in sorted(EXTERNAL_EMBEDDINGS_DIR.glob("*.npz")):
        name = path.stem.lower()
        Z, ids = load_external_embedding_npz(path)
        lookup = {sid: i for i, sid in enumerate(ids)}
        common = [sid for sid in sample_meta["sample_id"].astype(str) if sid in lookup]
        if len(common) < max(5, int(0.8 * len(sample_meta))):
            external_status[name] = {
                "status": "unavailable",
                "reason": f"Only {len(common)}/{len(sample_meta)} benchmark samples aligned",
                "path": str(path),
            }
            continue
        order = [lookup[sid] for sid in sample_meta["sample_id"].astype(str)]
        aligned = Z[order]
        benchmark_payloads[name] = aligned
        external_status[name] = {
            "status": "available",
            "path": str(path),
            "n_aligned_samples": int(len(order)),
            "dimension": int(aligned.shape[1]),
        }

print("Models entering benchmark:", list(benchmark_payloads))
print("External status:", json.dumps(external_status, indent=2))


## 13. Primary held-out results

Interpret these as **predictive representation evidence**, not evidence of causality, regenerative efficacy, or clinical utility.

The test set is determined entirely by the locked split.


In [ ]:
results_rows = []
full_results = {}

for name, Z in benchmark_payloads.items():
    res = evaluate_model(name, Z, sample_meta)
    full_results[name] = res
    row = {
        "model": name,
        "injury_auroc": res["injury"]["auroc"],
        "injury_auroc_ci_low": res["injury"]["test_auroc_ci_low"],
        "injury_auroc_ci_high": res["injury"]["test_auroc_ci_high"],
        "injury_auprc": res["injury"]["auprc"],
        "injury_balanced_accuracy": res["injury"]["balanced_accuracy"],
        "injury_f1_macro": res["injury"]["f1_macro"],
        "injury_permutation_p": res["injury"]["permutation_pvalue"],
        "maturation_mae": res["maturation"]["mae"],
        "maturation_rmse": res["maturation"]["rmse"],
        "maturation_r2": res["maturation"]["r2"],
        "maturation_r2_ci_low": res["maturation"]["r2_ci_low"],
        "maturation_r2_ci_high": res["maturation"]["r2_ci_high"],
        "test_samples": res["test_samples"],
        "test_species": ";".join(res["test_species"]),
        "test_studies": ";".join(res["test_studies"]),
    }
    results_rows.append(row)

results_df = pd.DataFrame(results_rows).sort_values("model")
results_df.to_csv(RESULTS_DIR / "benchmark_results.csv", index=False)
(RESULTS_DIR / "benchmark_results.json").write_text(
    json.dumps(full_results, indent=2, sort_keys=True, default=float) + "\n"
)
display(results_df)


## 14. Cross-study and species-unseen transfer audit

Cross-study transfer is represented by the locked test partition because entire study families are held out.

A test species is called **species-unseen** only if it is absent from the train + validation model-fitting population. Otherwise the notebook explicitly reports that the representation has seen that species during training and does not call this a cross-species transfer result.


In [ ]:
train_val_species = set(sample_meta.loc[sample_meta["split"].isin(["train", "validation"]), "species"])
test_species = set(sample_meta.loc[sample_meta["split"].eq("test"), "species"])
unseen_species = sorted(test_species - train_val_species)

species_transfer_report = {
    "train_validation_species": sorted(train_val_species),
    "test_species": sorted(test_species),
    "species_unseen_in_test": unseen_species,
    "status": "demonstrated" if unseen_species else "not_demonstrated",
    "note": (
        "Species-unseen transfer is valid only for species absent from model-fitting data."
        if unseen_species else
        "Every test species was represented in training/validation; do not label this as species-unseen transfer."
    ),
}
(RESULTS_DIR / "cross_species_report.json").write_text(
    json.dumps(species_transfer_report, indent=2, sort_keys=True) + "\n"
)
print(json.dumps(species_transfer_report, indent=2))

# Evaluate an unseen-species subset only when it exists.
species_subset_rows = []
if unseen_species:
    subset_mask = sample_meta["split"].eq("test") & sample_meta["species"].isin(unseen_species)
    subset_idx = np.flatnonzero(subset_mask.to_numpy())
    train_idx = np.flatnonzero(sample_meta["split"].isin(["train", "validation"]).to_numpy())
    if len(subset_idx) >= 4 and sample_meta.iloc[train_idx]["injury"].nunique() == 2 and sample_meta.iloc[subset_idx]["injury"].nunique() == 2:
        # Probe selection remains on train+validation; the unseen species is never used to fit it.
        for name, Z in benchmark_payloads.items():
            est = make_pipeline(
                StandardScaler(),
                LogisticRegression(C=1.0, max_iter=4000, random_state=SEED),
            )
            est.fit(Z[train_idx], sample_meta.iloc[train_idx]["injury"].to_numpy(dtype=int))
            score = est.predict_proba(Z[subset_idx])[:, 1]
            auc = roc_auc_score(sample_meta.iloc[subset_idx]["injury"].to_numpy(dtype=int), score)
            species_subset_rows.append({
                "model": name,
                "species_unseen_auroc": float(auc),
                "n_samples": int(len(subset_idx)),
                "species": ";".join(unseen_species),
            })
species_transfer_df = pd.DataFrame(species_subset_rows)
species_transfer_df.to_csv(RESULTS_DIR / "species_unseen_results.csv", index=False)
display(species_transfer_df if len(species_transfer_df) else pd.DataFrame([{
    "status": species_transfer_report["status"],
    "message": species_transfer_report["note"]
}]))


## 15. Runtime, provenance and final validation report


In [ ]:
import subprocess, datetime

runtime = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": (
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        if torch.cuda.is_available() else None
    ),
    "repo_commit": subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
}
(RESULTS_DIR / "runtime.json").write_text(json.dumps(runtime, indent=2, sort_keys=True) + "\n")
subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    stdout=(RESULTS_DIR / "pip_freeze.txt").open("w"),
    check=True,
)

final_summary = {
    "run_id": RUN_ID,
    "status": "completed",
    "scientific_scope": [
        "locked hierarchical split",
        "train-only gene selection",
        "CardiLearn frozen representation",
        "sample-level held-out predictive evaluation",
        "PCA and autoencoder baselines",
        "bootstrap uncertainty",
        "permutation null",
        "cross-study transfer",
        "species-unseen audit",
    ],
    "model_parameters": int(model.parameter_count()),
    "selected_genes": int(len(selected_idx)),
    "cells": int(len(metadata)),
    "samples": int(len(sample_meta)),
    "train_samples": int((sample_meta["split"] == "train").sum()),
    "validation_samples": int((sample_meta["split"] == "validation").sum()),
    "test_samples": int((sample_meta["split"] == "test").sum()),
    "external_models": external_status,
    "cross_species": species_transfer_report,
    "scientific_limitations": [
        "Predictive performance is not causal evidence.",
        "No result implies regenerative efficacy or clinical utility.",
        "External models are only compared when caller-supplied embeddings align to the locked biological samples.",
        "A single locked test set should not be repeatedly tuned against.",
    ],
}
(RESULTS_DIR / "final_summary.json").write_text(
    json.dumps(final_summary, indent=2, sort_keys=True, default=float) + "\n"
)

report = f"""# CardiLearn v0.5 real-data validation report

Run: {RUN_ID}
Repository commit: {runtime["repo_commit"]}
Device: {runtime["cuda_device"] or runtime["platform"]}

## Data lock
- Expression shape: {list(map(int, X.shape))}
- Selected genes: {len(selected_idx)}
- Study families: {metadata["study_family_id"].nunique()}
- Subjects: {metadata["subject_id"].nunique()}
- Samples: {sample_meta.shape[0]}

## Benchmark
Primary endpoint: held-out injury AUROC at biological-sample level.
Secondary endpoint: maturation regression.

## Interpretation
This report contains held-out predictive representation results only. It is not evidence of causal mechanism, regenerative efficacy, or clinical utility.

Species-unseen status: {species_transfer_report["status"]}
"""
(RESULTS_DIR / "run_summary.md").write_text(report)

display(Markdown(report))
display(results_df)


## 16. Plot and package the lightweight validation outputs

Large raw datasets are **not** copied into the GitHub results package. The local Colab run directory retains the model checkpoint and all generated artifacts.


In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.copy().sort_values("injury_auroc", ascending=False)
plt.figure(figsize=(9, 5))
plt.bar(plot_df["model"], plot_df["injury_auroc"], yerr=[
    plot_df["injury_auroc"] - plot_df["injury_auroc_ci_low"],
    plot_df["injury_auroc_ci_high"] - plot_df["injury_auroc"]
], capsize=4)
plt.ylabel("Held-out injury AUROC")
plt.xlabel("Representation")
plt.title("Locked biological-sample test set")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "injury_auroc_test.png", dpi=160)
plt.show()

LIGHTWEIGHT = [
    "data_lock.json",
    "selected_genes.npz",
    "selected_genes.txt",
    "metadata_vocab.json",
    "training_history.json",
    "training_summary.json",
    "autoencoder_history.json",
    "sample_metadata.parquet",
    "benchmark_results.csv",
    "benchmark_results.json",
    "cross_species_report.json",
    "species_unseen_results.csv",
    "runtime.json",
    "pip_freeze.txt",
    "final_summary.json",
    "run_summary.md",
    "injury_auroc_test.png",
]
light_dir = RESULTS_DIR / "lightweight"
light_dir.mkdir(exist_ok=True)
for name in LIGHTWEIGHT:
    src = RESULTS_DIR / name
    if src.exists():
        shutil.copy2(src, light_dir / name)

zip_path = shutil.make_archive(str(RESULTS_DIR / f"{RUN_ID}-lightweight"), "zip", light_dir)
print("Lightweight archive:", zip_path)


## 17. Optional GitHub push of lightweight results

This cell does **not** push raw expression data or the full model checkpoint.

Set `PUSH_RESULTS=True` at the configuration cell to create a results branch and commit the lightweight provenance/results package. The notebook asks for a GitHub token only at runtime and does not print it.


In [ ]:
if PUSH_RESULTS:
    import getpass
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (repo write access): ")
    if not token:
        raise ValueError("No GitHub token supplied")

    # Authenticate gh without echoing the token.
    subprocess.run(
        ["gh", "auth", "login", "--with-token"],
        input=token.encode(),
        cwd=REPO_DIR,
        check=True,
    )
    branch = f"results/{RUN_ID}"
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-b", branch], check=True)

    target = REPO_DIR / "runs" / RUN_ID
    target.mkdir(parents=True, exist_ok=True)
    for name in LIGHTWEIGHT:
        src = RESULTS_DIR / name
        if src.exists():
            shutil.copy2(src, target / name)

    subprocess.run(["git", "-C", str(REPO_DIR), "add", str(target.relative_to(REPO_DIR))], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "commit", "-m", f"results: CardiLearn real-data validation {RUN_ID}"],
        check=True,
    )
    subprocess.run(["git", "-C", str(REPO_DIR), "push", "-u", "origin", branch], check=True)
    print("Pushed branch:", branch)
else:
    print("PUSH_RESULTS=False — artifacts remain local under", RESULTS_DIR)


## Stop condition

A completed notebook run gives you:

**data lock → leakage audit → train-only feature selection → T4 training → frozen embeddings → biological-sample benchmark → uncertainty → cross-study/species audit → provenance package.**

Do not interpret a better AUROC by itself as proof that CardiLearn has discovered a regeneration mechanism or has regenerative efficacy. The next scientific step after a clean benchmark is independent biological interpretation/validation.
